### Notebook 4: Model Training, Validation & Testing
**Objective:** Train candidate classifiers on balanced training data, evaluate performance on the validation set, test on holdout data, and export metrics.

In [17]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score



In [18]:
X_train = pd.read_csv("../data/X_train.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").values.ravel()
y_val = pd.read_csv("../data/y_val.csv").values.ravel()
y_test = pd.read_csv("../data/y_test.csv").values.ravel()

print(f"Data Loaded -> Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Data Loaded -> Train: (7244, 15) | Val: (1056, 15) | Test: (1057, 15)


In [19]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

val_results = []
fitted_models = {}

print("--- TRAINING MODELS & EVALUATING ON VALIDATION SET ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]

    val_results.append({
        'Model': name,
        'Val_Accuracy': accuracy_score(y_val, y_val_pred),
        'Val_Precision': precision_score(y_val, y_val_pred),
        'Val_Recall': recall_score(y_val, y_val_pred),
        'Val_F1': f1_score(y_val, y_val_pred),
        'Val_ROC_AUC': roc_auc_score(y_val, y_val_proba)
    })

val_df = pd.DataFrame(val_results)
display(val_df.round(4))

--- TRAINING MODELS & EVALUATING ON VALIDATION SET ---


,Model,Val_Accuracy,Val_Precision,Val_Recall,Val_F1,Val_ROC_AUC
0,Logistic Regression,0.7282,0.4921,0.7821,0.6041,0.8279
1,Decision Tree,0.7491,0.5176,0.7857,0.6241,0.8333
2,Random Forest,0.7434,0.5110,0.7464,0.6067,0.8255
3,Gradient Boosting,0.7491,0.5173,0.8000,0.6283,0.8354


In [20]:
# Select final model based on validation performance
# (F1 favored over accuracy since the original churn distribution is imbalanced)
best_model_name = val_df.sort_values(by='Val_F1', ascending=False).iloc[0]['Model']
best_model = fitted_models[best_model_name]
print(f"Selected model (highest Val_F1): {best_model_name}")

print(f"--- EVALUATING FINAL MODEL ({best_model_name}) ON UNSEEN TEST SET ---")
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

test_results = [{
    'Model': best_model_name,
    'Test_Accuracy': accuracy_score(y_test, y_test_pred),
    'Test_Precision': precision_score(y_test, y_test_pred),
    'Test_Recall': recall_score(y_test, y_test_pred),
    'Test_F1': f1_score(y_test, y_test_pred),
    'Test_ROC_AUC': roc_auc_score(y_test, y_test_proba)
}]
test_df = pd.DataFrame(test_results)
display(test_df.round(4))

Selected model (highest Val_F1): Gradient Boosting
--- EVALUATING FINAL MODEL (Gradient Boosting) ON UNSEEN TEST SET ---


,Model,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC
0,Gradient Boosting,0.7606,0.5359,0.7438,0.623,0.8286


In [21]:
summary_df = val_df.merge(test_df, on='Model', how='left')
os.makedirs("../data", exist_ok=True)
RESULTS_PATH = "../data/model_results.csv"
summary_df.to_csv(RESULTS_PATH, index=False)
print(f"Results summary successfully exported to '{RESULTS_PATH}'!")

Results summary successfully exported to '../data/model_results.csv'!


In [22]:
os.makedirs("../models", exist_ok=True)

for name, model in fitted_models.items():
    joblib.dump(model, f"../models/{name.replace(' ', '_').lower()}.joblib")

joblib.dump(best_model_name, "../models/best_model_name.joblib")
np.save("../data/y_test_pred.npy", y_test_pred)
np.save("../data/y_test_proba.npy", y_test_proba)

print(f"Saved {len(fitted_models)} models and final predictions for NB5.")

Saved 4 models and final predictions for NB5.
